# 05 - Strategy 2: leave one field out (Table 2)

Every field is held out in turn, and the model is trained on the other fields of the same group
(ASP: 12 fields, BAU: 6, ASP+BAU: 18). The predictions for all held-out fields are pooled and scored once.

Within each fold:

- **Year feature.** The year is added as a numeric feature.
- **Per-year scaling.** Features are scaled with statistics from the training fields only. A global
  standardisation is applied first, then a per-year standardisation on top of it, using that year's raw
  means and standard deviations. The same transform is applied to the held-out field.
- **Tuning.** A random 80% of the training pixels is used for a randomised hyperparameter search
  (10 draws, 3-fold CV), and the model is fitted on that 80% with the best settings. The other 20% was set
  aside for early stopping, but the XGBoost version used for the paper (2.1) ignores early stopping passed
  through `fit`, so it is not used.

These steps are kept exactly as in the runs behind Table 2. Expect roughly an hour per group on a laptop.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

sys.path.append("../src")
from yieldml import CASES, TARGET, Y_MULT, load_all, scores
from scipy.stats import randint, uniform
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split

In [2]:
TABLE_DIR = Path("../results/tables")
data = load_all()

BASE_PARAMS = dict(objective="reg:squarederror", n_estimators=500, learning_rate=0.05, max_depth=5,
                   subsample=0.8, colsample_bytree=0.8, min_child_weight=3.0, reg_lambda=1.0,
                   reg_alpha=0.0, random_state=42, n_jobs=-1, tree_method="hist", eval_metric="rmse")
SEARCH_SPACE = {
    "max_depth": randint(3, 8),
    "learning_rate": uniform(0.02, 0.12),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": uniform(1.0, 8.0),
    "reg_lambda": uniform(0.0, 5.0),
    "reg_alpha": uniform(0.0, 1.0),
    "n_estimators": randint(500, 1500),
}

## Scaling and tuning

In [3]:
def year_scalers(train, cols):
    per_year = {}
    for yr, d in train.groupby("Year"):
        per_year[yr] = (d[cols].mean(), d[cols].std(ddof=0).replace(0, 1.0))
    overall = (train[cols].mean(), train[cols].std(ddof=0).replace(0, 1.0))
    return per_year, overall


def apply_scalers(df, cols, per_year, overall):
    out = df.copy()
    out[cols] = (out[cols] - overall[0]) / overall[1]
    # the per-year step works on the already standardised values (kept as in the published runs)
    for yr, (mean, std) in per_year.items():
        rows = out["Year"] == yr
        if rows.any():
            out.loc[rows, cols] = (out.loc[rows, cols] - mean) / std
    return out


def tuned_model(X, y):
    search = RandomizedSearchCV(XGBRegressor(**BASE_PARAMS), SEARCH_SPACE, n_iter=10,
                                scoring="neg_root_mean_squared_error",
                                cv=KFold(n_splits=3, shuffle=True, random_state=42),
                                refit=False, random_state=42, n_jobs=-1)
    try:
        search.fit(X, y)
        params = {**BASE_PARAMS, **search.best_params_}
    except Exception as e:
        print(f"search failed ({e}), using the base settings")
        params = BASE_PARAMS
    return XGBRegressor(**params).fit(X, y)

## Leave one field out

In [4]:
def leave_field_out(df):
    df = df.copy()
    df["Year_num"] = pd.to_numeric(df["Year"])
    df["_y"] = df[TARGET] * Y_MULT
    pooled = {case: ([], []) for case in CASES}

    for field in df["FieldName"].unique():
        train_all, test_all = df[df["FieldName"] != field], df[df["FieldName"] == field]
        for case, feats in CASES.items():
            feats = [f for f in feats if f in df.columns]
            cols = feats + ["Year_num"]
            tr = train_all.dropna(subset=cols + ["_y"])
            te = test_all.dropna(subset=cols + ["_y"])
            if tr.empty or te.empty:
                continue

            per_year, overall = year_scalers(tr, feats)
            X_tr = apply_scalers(tr, feats, per_year, overall)[cols]
            X_te = apply_scalers(te, feats, per_year, overall)[cols]

            X_fit, _, y_fit, _ = train_test_split(X_tr, tr["_y"].values, test_size=0.2, random_state=42)
            model = tuned_model(X_fit, y_fit)

            pooled[case][0].append(te["_y"].values)
            pooled[case][1].append(model.predict(X_te))
        print(f"  held out {field}")

    rows = []
    for case, (obs, pred) in pooled.items():
        if obs:
            rows.append({"Case": case, **scores(np.concatenate(obs), np.concatenate(pred))})
    return pd.DataFrame(rows).set_index("Case")

In [5]:
table2 = {}
for group in ["ASP+BAU", "ASP", "BAU"]:
    print(group)
    table2[group] = leave_field_out(data[group])
    table2[group].to_csv(TABLE_DIR / f"table2_leave_field_out_{group.replace('+', '_')}.csv")

table2 = pd.concat(table2, axis=1)
table2.to_csv(TABLE_DIR / "table2_leave_field_out.csv")
table2.round({(g, "R2"): 2 for g in table2.columns.levels[0]}
             | {(g, "RMSE"): 0 for g in table2.columns.levels[0]}
             | {(g, "RRMSE"): 1 for g in table2.columns.levels[0]})

ASP+BAU
  held out S2
  held out S4
  held out S5
  held out S6
  held out SB1
  held out SB4
  held out SB5
  held out SB7
  held out SCD2
  held out SCD3
  held out SCD5
  held out SCD6
  held out S3
  held out S7
  held out SB3
  held out SB6
  held out SCD4
  held out SCD7
ASP
  held out S2
  held out S4
  held out S5
  held out S6
  held out SB1
  held out SB4
  held out SB5
  held out SB7
  held out SCD2
  held out SCD3
  held out SCD5
  held out SCD6
BAU
  held out S3
  held out S7
  held out SB3
  held out SB6
  held out SCD4
  held out SCD7


ASP+BAU                 ASP                BAU              
           R2    RMSE RRMSE    R2   RMSE RRMSE    R2    RMSE RRMSE
Case                                                              
Case1    0.54  1164.0  32.8  0.75  960.0  27.0  0.53  1030.0  29.1
Case2    0.53  1181.0  33.3  0.73  983.0  27.6  0.52  1047.0  29.5
Case3    0.58  1112.0  31.3  0.73  982.0  27.6  0.61   941.0  26.5
Case4    0.58  1115.0  31.4  0.75  944.0  26.5  0.60   957.0  27.0
Case5    0.53  1186.0  33.4  0.74  973.0  27.4  0.50  1070.0  30.2
Case6    0.58  1117.0  31.5  0.74  965.0  27.1  0.59   966.0  27.3
Case7    0.56  1145.0  32.2  0.74  975.0  27.4  0.56   997.0  28.1
Case8    0.52  1194.0  33.6  0.78  891.0  25.0  0.32  1246.0  35.2